# ESG Investment Intelligence: Data-Driven Sustainable Investing

This notebook demonstrates the ideas from the **ESG Investment Intelligence** research paper: using verified environmental data for portfolio analysis, sector benchmarking, and climate risk assessment. It uses the **Eko Client** to access satellite-verified emissions data (Climate TRACE) and related ESG data.

**Key themes:** Beyond disclosure (verified data), portfolio carbon footprint, sector benchmarking, geographic risk, and the risk lens (regulatory, market, physical, transition).

---

**If the kernel died:** Use **Kernel → Restart Kernel** (or Restart from the toolbar). Then run cells from the top. If Jupyter itself quit, restart it (e.g. `jupyter notebook` or reopen from Cursor) and reopen this notebook.

## Setup and Authentication

**Important:** Before running the notebook, install the required dependencies:

```bash
pip install httpx>=0.24.0 pydantic>=2.0.0 typing-extensions>=4.5.0
```

Or install from the eko-client requirements file:

```bash
pip install -r eko-client-python/requirements.txt
```

Then import necessary packages.

**Kernel:** Use the project venv so dependencies (httpx, pandas, etc.) are available.

1. **Select kernel:** Click the kernel name in the **top-right**. Choose **"Jana (.venv Python 3.13)"** or **".venv (Python 3.13.9)"**. (If you don't see it, run in terminal: `cd /Users/willardmechem/Projects/repos/Jana && .venv/bin/python -m ipykernel install --user --name jana-venv --display-name "Jana (.venv Python 3.13)"` then refresh the kernel list.)
2. **ModuleNotFoundError: No module named 'httpx'?** The wrong kernel is selected — switch to **Jana (.venv Python 3.13)** and run the cells again.
3. **Run a cell:** Click the cell, then **Shift+Enter**.

In [5]:
# Kernel test: if you see "Kernel OK" below, the kernel is running. If not, select a kernel (top-right).
print("Kernel OK")

Kernel OK


In [6]:
import osimport warningsfrom eko_client import EkoUserClient# Suppress Jupyter introspection warnings for async methodswarnings.filterwarnings('ignore', message='coroutine.*was never awaited')# Get credentials from environmentBASE_URL = os.environ.get("JANA_API_URL", "https://api-dev.jana.earth")USERNAME = os.environ.get("JANA_USERNAME", "dev-user")PASSWORD = os.environ.get("JANA_PASSWORD", "")if not PASSWORD:    from getpass import getpass    PASSWORD = getpass(f"Enter password for {USERNAME}: ")# Initialize clientclient = EkoUserClient(    base_url=BASE_URL,    username=USERNAME,    password=PASSWORD,    timeout=60)# Test connectiontry:    health = client.get_health()    print(f"✅ Connected to Jana Earth API: {health.get('status', 'unknown')}")    print(f"   API Version: {health.get('version', 'unknown')}")except Exception as e:    print(f"❌ Connection failed: {e}")    print("   Please check your credentials and ensure the API is running.")

Running...
Path: /Users/willardmechem/Projects/repos/Jana/eko-client-python
Path and EkoUserClient OK


In [7]:
# Cell 2: Pandas and plotting (run after Cell 1)
import os
import json
from datetime import datetime, timedelta
from typing import Optional, List, Dict, Any
import pandas as pd
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import seaborn as sns

pd.set_option("display.max_columns", None)
pd.set_option("display.max_rows", 100)
pd.set_option("display.float_format", "{:,.2f}".format)
sns.set_style("whitegrid")
plt.rcParams["figure.figsize"] = (12, 6)
print("Pandas and plotting OK")

Pandas and plotting OK


In [ ]:
# Helper function for pagination
import time

def fetch_all_pages(client_method, **kwargs):
    """Fetch all pages of data using pagination. Returns list of all results."""
    start_time = time.time()
    all_results = []
    offset = 0
    limit = kwargs.pop('limit', 10000)
    max_limit = 10000
    api_page_limit = 100
    if limit is None:
        limit = max_limit
    page = 1
    total_fetched = 0
    while True:
        try:
            kwargs['offset'] = offset
            kwargs['limit'] = min(limit, max_limit)
            response = client_method(**kwargs)
            if not response or 'results' not in response:
                break
            results = response.get('results', [])
            if not results:
                break
            all_results.extend(results)
            total_fetched += len(results)
            total_count = response.get('count', 0)
            if total_count > 0 and total_fetched >= total_count:
                break
            if len(results) < api_page_limit:
                break
            offset += len(results)
            page += 1
            if page > 1000:
                break
        except Exception as e:
            print(f"   Error fetching page {page}: {e}")
            raise
    duration = time.time() - start_time
    if duration > 0:
        print(f"   Total records fetched: {len(all_results):,} in {duration:.2f}s")
    return all_results

## ESG Analysis Configuration

Demo portfolio / analysis settings: sectors and date range for the following sections. Adjust as needed for your use case.

In [ ]:
# Demo portfolio / analysis config
# Sectors to focus on (names must match API; we'll fetch available sectors if needed)
DEMO_SECTORS = ['Power', 'Manufacturing', 'Oil and gas']  # Example; adjust after get_climatetrace_sectors()
# Date range: last 12 months (optional filter for emissions)
DATE_TO = datetime.utcnow()
DATE_FROM = DATE_TO - timedelta(days=365)
print(f"Date range: {DATE_FROM.date()} to {DATE_TO.date()}")
print(f"Demo sectors: {DEMO_SECTORS}")

Date range: 2025-02-04 to 2026-02-04
Demo sectors: ['Power', 'Manufacturing', 'Oil and gas']


/var/folders/g0/mvzztl75695dsrkdwqn0tz8c0000gn/T/ipykernel_80792/2466171491.py:5: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  DATE_TO = datetime.utcnow()


---
## 1. Beyond Disclosure: Verified Data

Traditional ESG analysis relies on corporate self-disclosure. Our approach uses **satellite-verified emissions** with **asset-level granularity** and **monthly updates**—enabling assessment based on observed reality, not just stated intentions. This section shows platform coverage and data availability.

In [ ]:
# Safe display (works when kernel restarts or display is missing)
def _show(df):
    try:
        display(df)
    except NameError:
        print(df.to_string() if hasattr(df, 'to_string') else df)

# Platform summary and Climate TRACE coverage (API failures raise; fix and re-run)
summary = client.get_summary()
if isinstance(summary, dict):
    print("Platform summary:")
    for k, v in summary.items():
        if k != 'table_summaries' and not isinstance(v, (dict, list)):
            print(f"   {k}: {v}")
    if 'platform_overview' in summary:
        po = summary['platform_overview']
        if isinstance(po, dict):
            for k, v in po.items():
                print(f"   {k}: {v}")
else:
    print("Summary:", summary)

totals = client.get_climatetrace_emissions_totals()
print("\nClimate TRACE emissions totals:")
if isinstance(totals, dict):
    for k, v in totals.items():
        if v is not None:
            print(f"   {k}: {v:,.0f}" if isinstance(v, (int, float)) else f"   {k}: {v}")
date_range = client.get_climatetrace_emissions_date_range()
if isinstance(date_range, dict):
    print(f"   Date range: {date_range.get('min_start_time')} to {date_range.get('max_start_time')}")

assets_resp = client.get_climatetrace_assets(limit=100, offset=0)
assets_list = assets_resp.get('results', []) if isinstance(assets_resp, dict) else (assets_resp if isinstance(assets_resp, list) else [])
if assets_list:
    df_assets = pd.DataFrame(assets_list)
    print(f"\nAsset-level sample: {len(df_assets)} assets")
    _show(df_assets.head(10))
else:
    print("\nNo assets returned (empty or different response shape).")

get_summary failed: HTTP request failed: 
Climate TRACE totals/date_range failed: HTTP request failed: 


KeyboardInterrupt: 

## 2. Portfolio Carbon Footprint

**The question:** What is the total emissions exposure of my portfolio?

By mapping portfolio holdings (here: a subset of sectors or countries) to emissions data, we can calculate total CO2e attributable to the portfolio and optionally emissions intensity (e.g. tonnes per million dollars AUM).

In [ ]:
# Get sector totals and sum CO2e for "demo portfolio" (selected sectors)
sector_totals = client.get_climatetrace_emissions_sector_totals()
# API may return list directly or dict with 'results'
sector_list = sector_totals if isinstance(sector_totals, list) else sector_totals.get('results', [])
if not sector_list and isinstance(sector_totals, dict):
    sector_list = sector_totals.get('data', [])

df_sectors = pd.DataFrame(sector_list) if sector_list else pd.DataFrame()
if df_sectors.empty:
    print("No sector totals returned.")
else:
    # Filter to demo sectors if they exist
    if 'sector_name' in df_sectors.columns:
        portfolio_df = df_sectors[df_sectors['sector_name'].isin(DEMO_SECTORS)] if DEMO_SECTORS else df_sectors.head(5)
        if portfolio_df.empty:
            portfolio_df = df_sectors.head(3)  # Fallback: top 3 sectors
    else:
        portfolio_df = df_sectors.head(3)
    total_co2e = portfolio_df['total_co2e'].sum() if 'total_co2e' in portfolio_df.columns else 0
    print(f"Portfolio (selected sectors) total CO2e: {total_co2e:,.0f} tonnes")
    print("Emissions intensity: tonnes per $M AUM = total_co2e / (portfolio_AUM_millions); plug in your AUM.")
    display(portfolio_df)

## 3. Sector Benchmarking

**The question:** How does a company or sector compare to its peers?

With sector-level emissions data, we can compare a given sector (e.g. Power, Manufacturing) to the sector average and identify leaders versus laggards.

In [ ]:
# Sector totals: table and bar chart
sector_totals = client.get_climatetrace_emissions_sector_totals()
sector_list = sector_totals if isinstance(sector_totals, list) else sector_totals.get('results', sector_totals.get('data', []))
df_s = pd.DataFrame(sector_list) if sector_list else pd.DataFrame()

if not df_s.empty and 'total_co2e' in df_s.columns and 'sector_name' in df_s.columns:
    df_s = df_s.sort_values('total_co2e', ascending=False)
    avg_co2e = df_s['total_co2e'].mean()
    print(f"Sector average total CO2e: {avg_co2e:,.0f} tonnes")
    # Compare first sector (e.g. Power) to average
    if len(df_s) > 0:
        top_sector = df_s.iloc[0]
        print(f"Top sector '{top_sector['sector_name']}': {top_sector['total_co2e']:,.0f} tonnes (vs average {avg_co2e:,.0f})")
    display(df_s.head(15))
    # Bar chart (top N sectors)
    n_show = min(15, len(df_s))
    fig, ax = plt.subplots(figsize=(10, 5))
    df_plot = df_s.head(n_show)
    ax.barh(df_plot['sector_name'].astype(str), df_plot['total_co2e'], color='steelblue', alpha=0.8)
    ax.axvline(avg_co2e, color='red', linestyle='--', label=f'Average ({avg_co2e:,.0f})')
    ax.set_xlabel('Total CO2e (tonnes)')
    ax.set_ylabel('Sector')
    ax.set_title('Emissions by sector (benchmark: average)')
    ax.legend()
    plt.tight_layout()
    plt.show()
else:
    print("No sector totals available for benchmarking.")

## 4. Geographic Risk Assessment

**The question:** Where is climate-related risk concentrated?

Country-level emissions mapping reveals exposure to high-emission regions facing regulatory pressure, supply chain emissions risk, and transition risk as economies decarbonize at different paces.

In [ ]:
# Country totals: bar chart (top N countries)
country_totals = client.get_climatetrace_emissions_country_totals()
country_list = country_totals if isinstance(country_totals, list) else country_totals.get('results', country_totals.get('data', []))
df_c = pd.DataFrame(country_list) if country_list else pd.DataFrame()

if not df_c.empty and 'total_co2e' in df_c.columns:
    df_c = df_c.sort_values('total_co2e', ascending=False)
    label_col = 'country_name' if 'country_name' in df_c.columns else 'country_iso3'
    display(df_c.head(20))
    n_show = min(20, len(df_c))
    fig, ax = plt.subplots(figsize=(10, 6))
    df_plot = df_c.head(n_show)
    ax.barh(df_plot[label_col].astype(str), df_plot['total_co2e'], color='darkgreen', alpha=0.7)
    ax.set_xlabel('Total CO2e (tonnes)')
    ax.set_ylabel('Country')
    ax.set_title('Emissions by country (geographic risk concentration)')
    plt.tight_layout()
    plt.show()
else:
    print("No country totals available.")

## 5. The Risk Lens

Environmental data is not only about impact—it is about **risk**. Companies with high emissions face multiple pressures:

- **Regulatory risk:** Carbon pricing, emissions caps, and reporting requirements are expanding globally. High-emission facilities face direct cost impacts as regulations tighten.
- **Market risk:** Consumer preferences and B2B procurement increasingly favor low-carbon options. High emitters may lose market share to cleaner competitors.
- **Physical risk:** The same emissions driving climate change create physical risks—extreme weather, supply chain disruptions, asset damage—that affect high emitters.
- **Transition risk:** As the economy decarbonizes, assets and business models dependent on fossil fuels face stranding risk. Our data helps identify exposure.

Below: emissions by sector as a proxy for **transition risk exposure** (high-emission sectors face greater transition pressure).

In [ ]:
# Transition risk exposure by sector (reuse sector totals)
sector_totals = client.get_climatetrace_emissions_sector_totals()
sector_list = sector_totals if isinstance(sector_totals, list) else sector_totals.get('results', sector_totals.get('data', []))
df_risk = pd.DataFrame(sector_list) if sector_list else pd.DataFrame()

if not df_risk.empty and 'total_co2e' in df_risk.columns and 'sector_name' in df_risk.columns:
    df_risk = df_risk.sort_values('total_co2e', ascending=False).head(12)
    fig, ax = plt.subplots(figsize=(10, 5))
    ax.barh(df_risk['sector_name'].astype(str), df_risk['total_co2e'], color='coral', alpha=0.8)
    ax.set_xlabel('Total CO2e (tonnes)')
    ax.set_ylabel('Sector')
    ax.set_title('Transition risk exposure by sector (emissions)')
    plt.tight_layout()
    plt.show()
else:
    print("No sector data for risk chart.")

## 6. Compliance and the Alpha Opportunity

### Compliance dimension

Regulatory requirements for climate disclosure are accelerating:

- **TCFD** (Task Force on Climate-related Financial Disclosures)
- **CSRD** (Corporate Sustainability Reporting Directive, Europe)
- **SEC Climate Rules** (pending, United States)
- **ISSB Standards** (International Sustainability Standards Board)

All of these frameworks require emissions data that our platform provides. For regulated institutions, access to verified, asset-level emissions data is becoming mandatory.

### Alpha opportunity

Environmental data is not only about risk avoidance—it is about **identifying opportunity**:

- Companies successfully reducing emissions may outperform as markets price in transition value.
- Sectors enabling decarbonization (renewables, efficiency, electrification) offer growth exposure.
- Regions leading the transition may attract capital and talent.

Our data helps identify these opportunities before they are fully reflected in market prices.

In [ ]:
# Optional: platform definitions (sources, parameters, frameworks if exposed)
definitions = client.get_definitions()
if isinstance(definitions, dict):
    print("Platform definitions (keys):", list(definitions.keys()))
    for k in ['sources', 'parameters', 'frameworks']:
        if k in definitions and definitions[k]:
            print(f"\n{k}:", definitions[k][:3] if isinstance(definitions[k], list) else definitions[k])
else:
    print("Definitions:", type(definitions))